# 🌾 Train mô hình nhận diện bệnh/sâu hại lúa (Google Colab GPU)

**Trước khi chạy:**
1. Bật GPU: *Runtime → Change runtime type → Hardware accelerator = GPU (T4)*.
2. Nén dataset ở MÁY (dùng Python cho đúng mã tiếng Việt), rồi upload `rice_images.zip` lên **MyDrive**:
   ```bash
   python -c "import shutil; shutil.make_archive(r'C:\\Users\\pc\\Documents\\Data\\rice_images','zip',r'C:\\Users\\pc\\Documents\\Data\\rice_images')"
   ```
   → tạo `C:\Users\pc\Documents\Data\rice_images.zip` (train/validation/test ở gốc zip). Kéo file này vào Google Drive (MyDrive).
3. Chạy lần lượt các cell bên dưới. Cuối cùng tải `plant_disease.keras` + `labels.json` về, bỏ vào `ml-service/model/`.

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))
# Nếu dòng GPU rỗng -> Runtime > Change runtime type > GPU, rồi chạy lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, time

ZIP_PATH = '/content/drive/MyDrive/rice_images.zip'   # chỉnh nếu bạn để nơi khác
DATA_DIR = '/content/rice_images'

assert os.path.exists(ZIP_PATH), f'Khong thay {ZIP_PATH} — hay upload rice_images.zip len MyDrive'
if not os.path.isdir(os.path.join(DATA_DIR, 'train')):
    t = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print('Giai nen xong trong', round(time.time() - t), 'giay')

# Tự dò thư mục chứa train/validation (phòng khi zip lồng thêm 1 cấp)
if not os.path.isdir(os.path.join(DATA_DIR, 'train')):
    for root, dirs, _ in os.walk(DATA_DIR):
        if 'train' in dirs and 'validation' in dirs:
            DATA_DIR = root
            break
print('DATA_DIR =', DATA_DIR)
print('Thu muc con:', sorted(os.listdir(DATA_DIR)))

In [ ]:
# Bảng tên tiếng Việt (đồng bộ với ml-service/labels.py). KEY = tên thư mục lớp nguyên văn.
RICE_LABELS = {'Lúa - Bệnh Bạc lá (Bacterial leaf blight)': {'display': 'Lúa - Bệnh bạc lá (vi khuẩn)', 'healthy': False},
 'Lúa - Bệnh Cháy lá ( Leaf scald )': {'display': 'Lúa - Bệnh cháy lá (leaf scald)', 'healthy': False},
 'Lúa - Bệnh Gạch Nâu ( Narrow Brown Spot )': {'display': 'Lúa - Bệnh gạch nâu (đốm nâu hẹp)', 'healthy': False},
 'Lúa - Bệnh Than Vàng ( False Smut )': {'display': 'Lúa - Bệnh than vàng (false smut)', 'healthy': False},
 'Lúa - Bệnh lúa von (lúa đực) (Rice bakanae disease)': {'display': 'Lúa - Bệnh lúa von (mạ đực, bakanae)',
                                                         'healthy': False},
 'Lúa - Bệnh Đạo ôn lá và cổ bông (Rice leaf and neck blast)': {'display': 'Lúa - Bệnh đạo ôn (lá & cổ bông)',
                                                                'healthy': False},
 'Lúa - Bệnh Đốm Nâu ( Brown spot )': {'display': 'Lúa - Bệnh đốm nâu', 'healthy': False},
 'Lúa - Bệnh Đốm Vằn ( Khô vằn ) ( Sheath Blight )': {'display': 'Lúa - Bệnh đốm vằn (khô vằn)', 'healthy': False},
 'Lúa - Bọ trĩ ( Thrips )': {'display': 'Lúa - Bọ trĩ', 'healthy': False},
 'Lúa - Khỏe mạnh': {'display': 'Lúa - Khỏe mạnh', 'healthy': True},
 'Lúa - K': {'display': 'Lúa - Thiếu Kali (K)', 'healthy': False},
 'Lúa - N': {'display': 'Lúa - Thiếu đạm (N)', 'healthy': False},
 'Lúa - P': {'display': 'Lúa - Thiếu lân (P)', 'healthy': False},
 'Lúa - Rầy Nâu (Brown Plant Hopper)': {'display': 'Lúa - Rầy nâu', 'healthy': False},
 'Lúa - Sâu cuốn lá lớn ( Rice skipper)': {'display': 'Lúa - Sâu cuốn lá lớn', 'healthy': False},
 'Lúa - Sâu cuốn lá nhỏ ( Rice leaf folder)': {'display': 'Lúa - Sâu cuốn lá nhỏ', 'healthy': False},
 'Lúa - Sâu gai ( Hispa )': {'display': 'Lúa - Sâu gai (hispa)', 'healthy': False},
 'Lúa - Sâu năn ( Muỗi hành )(Rice Gall Midge)': {'display': 'Lúa - Sâu năn (muỗi hành)', 'healthy': False},
 'Lúa - Sâu đục thân (Rice yellow stem borer)': {'display': 'Lúa - Sâu đục thân (bướm 2 chấm)', 'healthy': False},
 'Lúa - Sâu đục thân (Sọc nâu )( Chilo suppressalis )': {'display': 'Lúa - Sâu đục thân sọc nâu', 'healthy': False},
 'Lúa - Tungro virus': {'display': 'Lúa - Bệnh vàng lùn Tungro (virus)', 'healthy': False},
 'Ngô - Bệnh cháy lá (Blight)': {'display': 'Ngô - Bệnh cháy lá (Northern Leaf Blight)', 'healthy': False},
 'Ngô - Bệnh gỉ sắt (Common Rust)': {'display': 'Ngô - Bệnh gỉ sắt', 'healthy': False},
 'Ngô - Bệnh đốm lá xám (Gray Leaf Spot)': {'display': 'Ngô - Bệnh đốm lá xám', 'healthy': False},
 'Ngô - Khỏe mạnh': {'display': 'Ngô - Khỏe mạnh', 'healthy': True},
 'Sầu riêng - Bệnh tảo (Algal)': {'display': 'Sầu riêng - Bệnh đốm tảo', 'healthy': False},
 'Sầu riêng - Bệnh cháy lá (Blight)': {'display': 'Sầu riêng - Bệnh cháy lá', 'healthy': False},
 'Sầu riêng - Khỏe mạnh': {'display': 'Sầu riêng - Khỏe mạnh', 'healthy': True},
 'Sầu riêng - Bệnh Phomopsis': {'display': 'Sầu riêng - Bệnh đốm lá Phomopsis', 'healthy': False},
 'Sầu riêng - Bệnh Rhizoctonia': {'display': 'Sầu riêng - Bệnh cháy lá Rhizoctonia', 'healthy': False}}

def display_name(label):
    info = RICE_LABELS.get(label)
    return info['display'] if info else label

# Kiểm tra tên lớp khớp (phát hiện lỗi mã hoá tên thư mục khi nén/giải nén)
import os
detected = set(os.listdir(os.path.join(DATA_DIR, 'train')))
expected = set(RICE_LABELS.keys())
missing, extra = expected - detected, detected - expected
print('So lop phat hien:', len(detected))
if missing: print('⚠️ THIEU so voi ky vong:', missing)
if extra:   print('⚠️ THUA/khac ten:', extra)
if not missing and not extra:
    print(f'✅ {len(detected)} lop khop chinh xac — labels se map dung PestEntry')

In [ ]:
import tensorflow as tf, numpy as np, os
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

mixed_precision.set_global_policy('mixed_float16')   # TĂNG TỐC: FP16 tensor core trên T4 (~1.5-2x)

IMG = 224            # nhanh hơn nữa: đổi 160 (~2x, giảm chút accuracy)
BATCH = 64           # batch lớn tận dụng GPU
EPOCHS = 12
FINE_TUNE = 3
AUTOTUNE = tf.data.AUTOTUNE

train_dir = os.path.join(DATA_DIR, 'train')
val_dir   = os.path.join(DATA_DIR, 'validation')

train_raw = tf.keras.utils.image_dataset_from_directory(train_dir, image_size=(IMG, IMG), batch_size=BATCH, label_mode='int')
val_raw   = tf.keras.utils.image_dataset_from_directory(val_dir,   image_size=(IMG, IMG), batch_size=BATCH, label_mode='int')
class_names = train_raw.class_names
print('So lop:', len(class_names))

aug = tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.1), layers.RandomZoom(0.1)])
# TĂNG TỐC: cache ảnh đã resize (uint8) ra đĩa -> epoch 2+ bỏ qua giải mã JPEG, nhanh gấp nhiều lần
def prep(ds, training, cache_path):
    ds = ds.map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=AUTOTUNE)
    ds = ds.cache(cache_path)
    if training:
        ds = ds.map(lambda x, y: (aug(tf.cast(x, tf.float32), training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)
train_ds = prep(train_raw, True,  '/content/cache_train')
val_ds   = prep(val_raw,   False, '/content/cache_val')

# class_weight bù các lớp ít ảnh (False Smut, lúa von ~70 ảnh)
counts = [len(os.listdir(os.path.join(train_dir, n))) for n in class_names]
total, k = sum(counts), len(class_names)
class_weight = {i: total / (k * max(c, 1)) for i, c in enumerate(counts)}

base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
model = models.Sequential([base, layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
                           layers.Dense(len(class_names), activation='softmax', dtype='float32')])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# TĂNG TỐC: dừng sớm khi val hết cải thiện, giữ trọng số tốt nhất
es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
print('>> Giai doan 1: train dau phan loai (base dong bang)')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, callbacks=[es])

if FINE_TUNE > 0:
    print('>> Giai doan 2: fine-tune 30 lop cuoi MobileNetV2')
    base.trainable = True
    for l in base.layers[:-30]:
        l.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE, class_weight=class_weight, callbacks=[es])

In [ ]:
# Đánh giá trên tập test (dùng cho báo cáo đồ án)
test_dir = os.path.join(DATA_DIR, 'test')
if os.path.isdir(test_dir):
    test_raw = tf.keras.utils.image_dataset_from_directory(test_dir, image_size=(IMG, IMG),
                                                           batch_size=BATCH, label_mode='int', shuffle=False)
    test_ds = test_raw.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
    loss, acc = model.evaluate(test_ds)
    print(f'>> Test accuracy: {acc:.4f}  |  loss: {loss:.4f}')
else:
    print('Khong co thu muc test/, bo qua.')

In [ ]:
import json, shutil
os.makedirs('/content/model', exist_ok=True)
MODEL  = '/content/model/plant_disease.keras'
LABELS = '/content/model/labels.json'

model.save(MODEL)
meta = [{'index': i, 'label': n, 'display': display_name(n)} for i, n in enumerate(class_names)]
with open(LABELS, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# Lưu sang Drive để không mất khi Colab ngắt
shutil.copy(MODEL,  '/content/drive/MyDrive/plant_disease.keras')
shutil.copy(LABELS, '/content/drive/MyDrive/labels.json')
print('✅ Da luu vao Drive: MyDrive/plant_disease.keras + labels.json')

# Tải trực tiếp về máy
from google.colab import files
files.download(MODEL)
files.download(LABELS)

## Sau khi tải xong
Copy 2 file vào thư mục model của service (tạo nếu chưa có):
```
ml-service/model/plant_disease.keras
ml-service/model/labels.json
```
Rồi chạy service ở máy:
```bash
cd ml-service
venv/Scripts/activate        # hoặc dùng venv/Scripts/python.exe
uvicorn app:app --port 8000
```
Mở http://localhost:8000/health thấy `"model_loaded": true` là xong. Báo lại để nối backend + test luồng thật.